<a href="https://www.kaggle.com/code/mariamali12/chromtransformer-architecture?scriptVersionId=315018336" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/mariamali12/chromteransformer-data/__results__.html
/kaggle/input/notebooks/mariamali12/chromteransformer-data/__notebook__.ipynb
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data.tar.gz
/kaggle/input/notebooks/mariamali12/chromteransformer-data/__output__.json
/kaggle/input/notebooks/mariamali12/chromteransformer-data/custom.css
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E061/classification/valid.csv
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E061/classification/train.csv
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E061/classification/test.csv
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E028/classification/valid.csv
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E028/classification/train.csv
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E028/classification/test.csv
/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E120

In [2]:
# Libraries for model architecture
import torch
import torch.nn as nn
# Libraries for dataset formatting and dataloader
from pathlib  import Path
from torch    import Tensor
from torch.utils.data import Dataset, DataLoader
# Libraries for Training
import time
import optuna
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, average_precision_score
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Model Architecture 

In [3]:
#1. Input projection
 
class InputProjection(nn.Module):
    """
    Input:  (batch, 100, 5)
    Output: (batch, 100, d_model)
    """
    def __init__(self, n_marks: int, d_model: int, dropout: float):
        super().__init__()
        self.proj    = nn.Linear(n_marks, d_model, bias=False)
        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, x):
        x = self.proj(x)      
        x = self.norm(x)
        x = self.dropout(x)
        return x

In [4]:
#2. Positional embedding 
class PositionalEmbedding(nn.Module):
    """
    Learned positional embeddings for 100 fixed bin positions. 
    Input:  (batch, 100, d_model)
    Output: (batch, 100, d_model)
    """
    def __init__(self, n_bins: int, d_model: int, dropout: float):
        super().__init__()
        self.embedding = nn.Embedding(n_bins, d_model)
        self.dropout   = nn.Dropout(dropout)
 
        positions = torch.arange(n_bins).unsqueeze(0)   
        self.register_buffer('positions', positions)
 
    def forward(self, x):
        pos_emb = self.embedding(self.positions)  
        #Merging content to location
        x = x + pos_emb                             
        x = self.dropout(x)
        return x
 

In [5]:
# 3. Transformer encoder block 
 
class EncoderBlock(nn.Module):
    """
    One Transformer encoder block.
    Input:  (batch, 100, d_model)
    Output: (batch, 100, d_model)
    """
    def __init__(self, d_model: int, n_heads: int, ffn_mult: int, dropout: float):
        super().__init__()
 
        assert d_model % n_heads == 0, (
            f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        )
 
        # Self-attention
        self.norm1   = nn.LayerNorm(d_model)
        self.attn    = nn.MultiheadAttention(
            embed_dim   = d_model,
            num_heads  = n_heads,
            dropout     = dropout,
            batch_first = True,     # (batch, seq, d_model)
        )
        self.drop1   = nn.Dropout(dropout)
 
        # Feed forward network
        self.norm2   = nn.LayerNorm(d_model)
        self.ffn     = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_mult),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ffn_mult, d_model),
        )
        self.drop2   = nn.Dropout(dropout)
 
    def forward(self, x, return_attn: bool = False):
        # Self attention sublayer
        residual = x
        x_norm   = self.norm1(x)
        attn_out, attn_weights = self.attn(
            query            = x_norm,
            key              = x_norm,
            value            = x_norm,
            need_weights     = return_attn,
            average_attn_weights = True,    
        )
        x = residual + self.drop1(attn_out)
 
        # Feed-forward sublayer 
        residual = x
        x = residual + self.drop2(self.ffn(self.norm2(x)))
 
        if return_attn:
            return x, attn_weights   # attn_weights: (batch, 100, 100)
        return x

In [6]:
# 4. Full ChromeTransformer 
 
class ChromeTransformer(nn.Module):
    """
    Full model: projection → positional embedding → N encoder blocks
                → mean pool → classifier.
 
    Args:
        n_marks : number of histone marks (5, fixed by the dataset)
        n_bins  : number of genomic bins  (100, fixed by the dataset)
        d_model : embedding dimension (hyperparameter)
        n_heads : number of attention heads (hyperparameter)
        n_layers: number of encoder blocks (hyperparameter)
        ffn_mult : FFN hidden dim = d_model × ffn_mult (default 4)
        dropout : dropout probability (hyperparameter)
        n_classes: output classes (2 for binary classification)
 
     """
    def __init__(
        self,
        n_marks  : int   = 5,
        n_bins   : int   = 100,
        d_model  : int   = 64,
        n_heads  : int   = 4,
        n_layers : int   = 2,
        ffn_mult : int   = 4,
        dropout  : float = 0.1,
        n_classes: int   = 2,
    ):
        super().__init__()
 
        self.n_bins   = n_bins
        self.d_model  = d_model
        self.n_layers = n_layers
 
        self.input_proj = InputProjection(n_marks, d_model, dropout)
        self.pos_emb    = PositionalEmbedding(n_bins, d_model, dropout)
 
        self.encoder    = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ffn_mult, dropout)
            for _ in range(n_layers)
        ])
 
        self.norm       = nn.LayerNorm(d_model)     
 
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_classes),
        )
 
    def forward(self, x, return_attn: bool = False):
        """
        x: (batch, 100, 5) — raw histone mark values per bin
        """
        # Embed 
        x = self.input_proj(x)   # (batch, 100, d_model)
        x = self.pos_emb(x)      # (batch, 100, d_model)
 
        # Encode
        attn_weights_all = []
        for block in self.encoder:
            if return_attn:
                x, attn_w = block(x, return_attn=True)
                attn_weights_all.append(attn_w)   # (batch, 100, 100)
            else:
                x = block(x)
 
        x = self.norm(x)         # (batch, 100, d_model)
 
        # Pool 
        x = x.mean(dim=1)        # average over bins
 
        # Classify 
        logits = self.classifier(x)              # (batch, 2)
 
        if return_attn:
            return logits, attn_weights_all
        return logits
 
 

# Dataset & Data Loader

In [7]:
#import a cell type train CSV 
E003_train = pd.read_csv('/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E003/classification/train.csv',header=None)
E003_train.head()

,0,1,2,3,4,5,6,7
0,3,1,0,1,0,2,0,1
1,3,2,1,2,2,8,0,1
2,3,3,0,5,2,6,1,1
3,3,4,1,2,5,2,1,1
4,3,5,2,2,1,16,1,1


CSV format (no header):
  - Col 0: GeneID   (int)
  - Col 1: BinID    (int, 1–100)
  - Col 2: H3K27me3 (float)
  - Col 3: H3K36me3 (float)
  - Col 4: H3K4me1  (float)
  - Col 5: H3K4me3  (float)
  - Col 6: H3K9me3  (float)
  - Col 7: Label    (int, 0 or 1)
 
Each gene occupies exactly 100 consecutive rows (one per bin), already
sorted by BinID within each gene block. The Dataset groups rows by GeneID
and returns one (100, 5) tensor per gene.

In [8]:
#Column indices (no header in CSV)
 
COL_GENE   = 0
COL_BIN    = 1
COL_MARKS  = [2, 3, 4, 5, 6]   # H3K27me3, H3K36me3, H3K4me1, H3K4me3, H3K9me3
COL_LABEL  = 7
# Features 
N_BINS     = 100
N_MARKS    = 5

In [9]:
#Dataset
 
class ChromeDataset(Dataset):
    """
    PyTorch Dataset for one split (train / valid / test) of one cell type.
    Args:
        csv_path : path to train.csv / valid.csv / test.csv
          """
 
    def __init__(self, csv_path: str | Path, normalize: bool = False):
        csv_path = Path(csv_path)
        assert csv_path.exists(), f"CSV not found: {csv_path}"
 
        #1. Read the CSV into a DataFrame.
        df = pd.read_csv(csv_path, header=None)
 
        assert df.shape[1] == 8, (
            f"Expected 8 columns, got {df.shape[1]}. "
            f"Check COL_* constants match your CSV format."
        )
 
        #2. Sort by (GeneID, BinID) to guarantee bin order within each gene —
        # The CSV is likely already sorted, but we enforce it to be safe.
        df = df.sort_values([COL_GENE, COL_BIN]).reset_index(drop=True)
 
        n_rows = len(df)
        assert n_rows % N_BINS == 0, (
            f"Row count {n_rows} is not divisible by N_BINS={N_BINS}. "
            f"Some genes may have missing or duplicate bin rows."
        )
        n_genes = n_rows // N_BINS
 
        # 3. Extract the mark values as a numpy array of shape (n_genes × 100, 5), then reshape to (n_genes, 100, 5).
       
        marks = df.iloc[:, COL_MARKS].values.astype(np.float32)  # (n_rows, 5)
        marks = marks.reshape(n_genes, N_BINS, N_MARKS)           # (n_genes, 100, 5)
 
        # 4. Extract labels and gene IDs (one per gene, taken from the first row of each 100-row block — all rows for a gene share the same label).
        # All 100 rows of a gene share the same label — take every 100th row.
        labels   = df.iloc[::N_BINS][COL_LABEL].values.astype(np.int64)   # (n_genes,)
        gene_ids = df.iloc[::N_BINS][COL_GENE].values.astype(np.int64)    # (n_genes,)
 
        assert len(labels) == n_genes, (
            f"Label count {len(labels)} != n_genes {n_genes}"
        )
 
        # Confirm labels are binary {0, 1}
        unique_labels = set(np.unique(labels).tolist())
        assert unique_labels.issubset({0, 1}), (
            f"Unexpected label values: {unique_labels}. Expected {{0, 1}}."
        )
 
        # Store as tensors
        self.X        = torch.from_numpy(marks)     # (n_genes, 100, 5)
        self.y        = torch.from_numpy(labels)    # (n_genes,)
        self.gene_ids = torch.from_numpy(gene_ids)  # (n_genes,)
        self.n_genes  = n_genes
 
    def __len__(self) -> int:
        return self.n_genes
 
    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor, Tensor]:
        return self.X[idx], self.y[idx], self.gene_ids[idx]
 
    def label_counts(self) -> dict:
        """Returns {0: count, 1: count} for quick balance check."""
        counts = self.y.bincount()
        return {0: counts[0].item(), 1: counts[1].item()}
 

In [10]:
# Loader factory
 
def get_loaders(
    cell_type_dir : str | Path,
    batch_size    : int  = 64,
    normalize     : bool = False,
    num_workers   : int  = 2,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """
    Returns (train_loader, valid_loader, test_loader) for one cell type.
 
    Args:
        cell_type_dir : path to the cell type directory, e.g.
                        "/kaggle/working/data/E047"
                        Expected to contain:
                          classification/train.csv
                          classification/valid.csv
                          classification/test.csv
        batch_size    : number of genes per batch (default 64)
        normalize     : passed through to ChromeDataset
        num_workers   : DataLoader worker processes (default 2)
    """
    cell_type_dir = Path(cell_type_dir)
    split_dir     = cell_type_dir / "classification"
 
    assert split_dir.exists(), (
        f"Expected subdirectory 'classification/' inside {cell_type_dir}, "
        f"but it was not found. Check your data path."
    )
 
    datasets = {
        split: ChromeDataset(split_dir / f"{split}.csv", normalize=normalize)
        for split in ["train", "valid", "test"]
    }
 
    loaders = {
        "train": DataLoader(
            datasets["train"],
            batch_size  = batch_size,
            shuffle     = True,       # shuffle genes each epoch
            num_workers = num_workers,
            pin_memory  = True,       # faster CPU→GPU transfer
        ),
        "valid": DataLoader(
            datasets["valid"],
            batch_size  = batch_size * 2,   # no gradients → larger batch is fine
            shuffle     = False,
            num_workers = num_workers,
            pin_memory  = True,
        ),
        "test": DataLoader(
            datasets["test"],
            batch_size  = batch_size * 2,
            shuffle     = False,
            num_workers = num_workers,
            pin_memory  = True,
        ),
    }
 
    return loaders["train"], loaders["valid"], loaders["test"]


In [11]:

# Sanity check
 
if __name__ == "__main__":
 
    import sys
 
    # Default to E047 if no path is provided
    CELL_TYPE_DIR = "/kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E047"

    print(f"Loading cell type: {CELL_TYPE_DIR}")
    print()
    
    train_loader, valid_loader, test_loader = get_loaders(
        cell_type_dir = CELL_TYPE_DIR,
        batch_size    = 64,
    )
    
    for name, loader in [("train", train_loader),
                          ("valid", valid_loader),
                          ("test",  test_loader)]:
        ds = loader.dataset
        lc = ds.label_counts()
        print(f"{name:>5}  |  genes: {len(ds):>6,}  |  "
              f"label 0: {lc[0]:>5,}  label 1: {lc[1]:>5,}  |  "
              f"balance: {lc[1]/(lc[0]+lc[1]):.1%} positive")
    
    print()
    
    X, y, gene_ids = next(iter(train_loader))
    
    print(f"Batch shapes:")
    print(f"  X        : {X.shape}")
    print(f"  y        : {y.shape}")
    print(f"  gene_ids : {gene_ids.shape}")
    print(f"\nX dtype    : {X.dtype}")
    print(f"y dtype    : {y.dtype}")
    print(f"\nX value range : [{X.min():.3f}, {X.max():.3f}]")
    print(f"y unique vals : {y.unique().tolist()}")
 
 
 
    # ── Confirm model accepts this batch ──
    sys.path.insert(0, "/kaggle/input/notebooks/mariamali12/chromteransformer-data/data")
    try:
        from chrome_transformer import ChromeTransformer
        model  = ChromeTransformer()
        logits = model(X)
        print(f"Model output shape : {logits.shape}  — expected torch.Size([64, 2])")
        print()
        print("Sanity check passed.")
    except ImportError:
        print("chrome_transformer.py not found — skipping model forward pass check.")
        print("Sanity check passed (data loader only).")


Loading cell type: /kaggle/input/notebooks/mariamali12/chromteransformer-data/data/E047

train  |  genes:  6,601  |  label 0: 3,752  label 1: 2,849  |  balance: 43.2% positive
valid  |  genes:  6,601  |  label 0: 4,157  label 1: 2,444  |  balance: 37.0% positive
 test  |  genes:  6,600  |  label 0: 4,709  label 1: 1,891  |  balance: 28.7% positive

Batch shapes:
  X        : torch.Size([64, 100, 5])
  y        : torch.Size([64])
  gene_ids : torch.Size([64])

X dtype    : torch.float32
y dtype    : torch.int64

X value range : [0.000, 246.000]
y unique vals : [0, 1]
chrome_transformer.py not found — skipping model forward pass check.
Sanity check passed (data loader only).


# Training

In [12]:
# 1. Config & paths
 
DATA_ROOT   = Path("/kaggle/input/notebooks/mariamali12/chromteransformer-data/data")
OUTPUT_DIR  = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(exist_ok=True)
 
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# Default hyperparameters — will be overridden by Optuna
DEFAULT_HP  = dict(
    d_model      = 64,
    n_heads      = 4,
    n_layers     = 2,
    ffn_mult     = 4,
    dropout      = 0.1,
    lr           = 1e-3,
    batch_size   = 64,
    weight_decay = 1e-5,
)
 
# Training settings
MAX_EPOCHS      = 100
PATIENCE        = 10     # early stopping patience
N_OPTUNA_TRIALS = 30     # number of Optuna trials per cell type
OPTUNA_CELL     = "E047" # cell type used for hyperparameter search
 

In [13]:
# 2. Training utilities
 
def get_class_weights(train_loader: DataLoader) -> torch.Tensor:
    """
    Computes inverse-frequency class weights from the training split.
    Returns a (2,) tensor on DEVICE for use with CrossEntropyLoss.
 
    Weight for class c = 1 / count(c), then normalised so weights sum to 2
    (keeps loss magnitude comparable to unweighted case).
    """
    counts = train_loader.dataset.label_counts()  # {0: n0, 1: n1}
    n0, n1 = counts[0], counts[1]
    w0, w1 = 1.0 / n0, 1.0 / n1
    # Normalise
    total  = w0 + w1
    w0, w1 = 2 * w0 / total, 2 * w1 / total
    return torch.tensor([w0, w1], dtype=torch.float32, device=DEVICE)
 
 
def train_epoch(
    model     : nn.Module,
    loader    : DataLoader,
    optimizer : torch.optim.Optimizer,
    criterion : nn.Module,
) -> float:
    """
    One full pass over the training set.
    Returns mean training loss for the epoch.
    """
    model.train()
    total_loss = 0.0
 
    for X, y, _ in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
 
        optimizer.zero_grad()
        logits = model(X)             # (batch, 2)
        loss   = criterion(logits, y)
        loss.backward()
 
        # Gradient clipping — prevents occasional large gradient spikes
        # that can destabilise Transformer training
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
 
        optimizer.step()
        total_loss += loss.item()
 
    return total_loss / len(loader)
 
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probs  = []
    all_labels = []

    for X, y, _ in loader:
        X, y   = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss   = criterion(logits, y)
        total_loss += loss.item()

        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auroc      = roc_auc_score(all_labels, all_probs)
    aupr       = average_precision_score(all_labels, all_probs)
    mean_loss  = total_loss / len(loader)

    return mean_loss, auroc, aupr 

In [14]:
# 3. Single cell type training 
 
def train_cell_type(
    cell_type : str,
    hp        : dict,
    verbose   : bool = True,
) -> tuple[float, float, nn.Module]:
    """
    Trains one ChromeTransformer on one cell type with given hyperparameters.
    Uses early stopping on validation AUROC.
   
    Returns (test_auroc, test_aupr, best_model).
    """
    cell_dir = DATA_ROOT / cell_type
 
    train_loader, valid_loader, test_loader = get_loaders(
        cell_type_dir = cell_dir,
        batch_size    = hp["batch_size"],
    )
 
    # Model
    model = ChromeTransformer(
        d_model  = hp["d_model"],
        n_heads  = hp["n_heads"],
        n_layers = hp["n_layers"],
        ffn_mult = hp["ffn_mult"],
        dropout  = hp["dropout"],
    ).to(DEVICE)
 
    # Loss: weighted for class imbalance
    weights   = get_class_weights(train_loader)
    criterion = nn.CrossEntropyLoss(weight=weights)
 
    # Optimiser
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = hp["lr"],
        weight_decay = hp["weight_decay"],
    )
 
    # Early stopping state
    best_val_auroc  = -1.0
    best_state      = None
    patience_count  = 0
 
    if verbose:
        print(f"\n{'─'*60}")
        print(f"Cell type: {cell_type}  |  device: {DEVICE}")
        print(f"HP: d_model={hp['d_model']}, n_heads={hp['n_heads']}, "
              f"n_layers={hp['n_layers']}, dropout={hp['dropout']:.3f}, "
              f"lr={hp['lr']:.5f}")
        print(f"{'─'*60}")
        print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  "
              f"{'Val AUROC':>10}  {'':>8}")
 
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss            = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_auroc, _ = evaluate(model, valid_loader, criterion)

 
        #Early stopping
        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
            marker         = "✓"
        else:
            patience_count += 1
            marker          = ""
 
        if verbose:
            print(f"{epoch:>6}  {train_loss:>10.4f}  {val_loss:>10.4f}  "
                  f"{val_auroc:>10.4f}  {marker}")
 
        if patience_count >= PATIENCE:
            if verbose:
                print(f"\nEarly stopping at epoch {epoch} "
                      f"(best val AUROC: {best_val_auroc:.4f})")
            break
 
    #Restore best weights and evaluate on test set
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, test_auroc, test_aupr = evaluate(model, test_loader, criterion)
    if verbose:
        print(f"\nTest AUROC: {test_auroc:.4f}")
    # Save model weights
    weights_dir = OUTPUT_DIR / "weights"
    weights_dir.mkdir(exist_ok=True)
    torch.save({
        'model_state_dict' : best_state,
        'hp'               : hp,
        'cell_type'        : cell_type,
        'test_auroc'       : test_auroc,
        'test_aupr'        : test_aupr,
    }, weights_dir / f"{cell_type}.pt")
 
    return test_auroc, test_aupr, model

In [15]:
#4. Optuna hyperparameter search
 
def optuna_search(cell_type: str = OPTUNA_CELL) -> dict:
    """
    Runs Bayesian hyperparameter search on one cell type (OPTUNA_CELL).
    Optimises for validation AUROC.
 
    Search space:
      d_model      : {32, 64, 128}         — must be divisible by n_heads
      n_heads     : {2, 4, 8}
      n_layers    : {1, 2, 3, 4}
      ffn_mult     : {2, 4}
      dropout      : [0.05, 0.5]
      lr           : [1e-4, 1e-2] (log scale)
      batch_size  : {32, 64, 128}
      weight_decay : [1e-6, 1e-3] (log scale)
 
    Returns the best hyperparameter dict found.
    """
 
    cell_dir = DATA_ROOT / cell_type
 
    def objective(trial: optuna.Trial) -> float:
 
        # 1. Always offer the exact same choices for d_model
        d_model = trial.suggest_categorical("d_model", [32, 64, 128])
        
        # 2. Always offer the exact same static choices for n_heads
        # Do NOT filter the list here
        n_heads = trial.suggest_categorical("n_heads", [2, 4, 8])
    
        # 3. Check for compatibility AFTER suggesting
        # If the combination is mathematically impossible, skip (prune) this trial
        max_heads = d_model // 16  # ensures head_dim >= 16
        if d_model % n_heads != 0 or n_heads > max_heads:
            # This tells Optuna "this combo is bad, don't run it, and try something else"
            raise optuna.exceptions.TrialPruned()

        n_layers   = trial.suggest_int("n_layers",   1,    4)
        ffn_mult   = trial.suggest_categorical("ffn_mult", [2, 4])
        dropout    = trial.suggest_float("dropout",  0.05, 0.5)
        lr         = trial.suggest_float("lr",       1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
 
        hp = dict(
            d_model      = d_model,
            n_heads      = n_heads,
            n_layers     = n_layers,
            ffn_mult     = ffn_mult,
            dropout      = dropout,
            lr           = lr,
            batch_size   = batch_size,
            weight_decay = weight_decay,
        )
 
        train_loader, valid_loader, _ = get_loaders(
            cell_type_dir = cell_dir,
            batch_size    = batch_size,
        )
 
        model = ChromeTransformer(
            d_model  = d_model,
            n_heads  = n_heads,
            n_layers = n_layers,
            ffn_mult = ffn_mult,
            dropout  = dropout,
        ).to(DEVICE)
 
        weights   = get_class_weights(train_loader)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
 
        best_val_auroc = -1.0
        patience_count = 0
 
        for epoch in range(1, MAX_EPOCHS + 1):
            train_epoch(model, train_loader, optimizer, criterion)
            _, val_auroc, _ = evaluate(model, valid_loader, criterion)
 
            if val_auroc > best_val_auroc:
                best_val_auroc = val_auroc
                patience_count = 0
            else:
                patience_count += 1
 
            # Optuna pruning — stops unpromising trials early
            trial.report(val_auroc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
 
            if patience_count >= PATIENCE:
                break
 
        return best_val_auroc
 
    # Suppress Optuna's per-trial logging — we print a summary instead
    optuna.logging.set_verbosity(optuna.logging.WARNING)
 
    study = optuna.create_study(
        direction = "maximize",
        pruner    = optuna.pruners.MedianPruner(n_warmup_steps=5),
        sampler   = optuna.samplers.TPESampler(seed=42),
    )
 
    print(f"Running Optuna search on {cell_type} "
          f"({N_OPTUNA_TRIALS} trials)...")
 
    t0 = time.time()
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    elapsed = time.time() - t0
 
    best = study.best_params
    print(f"\nOptuna complete in {elapsed/60:.1f} min")
    print(f"Best val AUROC : {study.best_value:.4f}")
    print(f"Best params    : {best}")
 
    return best

In [16]:
# 5. Full 56-cell-type run
 
def run_all_cell_types(hp: dict = None) -> pd.DataFrame:
    """
    Trains ChromeTransformer on all 56 cell types and returns a DataFrame
    of per-cell-type test AUROCs.
 
    Args:
        hp : hyperparameter dict. If None, uses DEFAULT_HP.
             Typically you pass the result of optuna_search() here.
 
    Saves results to OUTPUT_DIR/auroc_results.csv after every cell type
    so progress is not lost if the Kaggle session times out.
 
    Returns a DataFrame with columns [cell_type, test_auroc].
    """
    if hp is None:
        print("No hyperparameters provided — using DEFAULT_HP.")
        hp = DEFAULT_HP
 
    # Discover all cell type directories
    cell_types = sorted([d.name for d in DATA_ROOT.iterdir()
                         if d.is_dir() and (d / "classification" / "train.csv").exists()])
 
    print(f"Found {len(cell_types)} cell types.")
    print(f"HP: {hp}\n")
 
    results      = []
    results_path = OUTPUT_DIR / "auroc_results.csv"
    t_start      = time.time()
 
    for i, ct in enumerate(cell_types, 1):
        t0 = time.time()
        print(f"[{i:>2}/{len(cell_types)}] {ct}", end="  ", flush=True)
 
        try:
            
            test_auroc, test_aupr, _ = train_cell_type(ct, hp, verbose=False)
            status = f"AUROC: {test_auroc:.4f}"

        except Exception as e:
            test_auroc = float("nan")
            status     = f"ERROR: {e}"
 
        elapsed = time.time() - t0
        print(f"{status}  ({elapsed:.0f}s)")
 
        results.append({"cell_type": ct, "test_auroc": test_auroc, "test_aupr": test_aupr})
 
        # Save incrementally after every cell type
        pd.DataFrame(results).to_csv(results_path, index=False)
 
    # Summary
    df          = pd.DataFrame(results)
    valid_aurocs = df["test_auroc"].dropna()
    total_time   = time.time() - t_start
 
    print(f"\n{'='*50}")
    print(f"RESULTS — ChromeTransformer")
    print(f"{'='*50}")
    print(f"Cell types trained : {len(valid_aurocs)} / {len(cell_types)}")
    print(f"Mean test AUROC : {df['test_auroc'].dropna().mean():.4f}")
    print(f"Mean test AUPR  : {df['test_aupr'].dropna().mean():.4f}")
    print(f"Std                : {valid_aurocs.std():.4f}")
    print(f"Min                : {valid_aurocs.min():.4f}")
    print(f"Max                : {valid_aurocs.max():.4f}")
    print(f"Total time         : {total_time/60:.1f} min")
    print(f"\nResults saved to   : {results_path}")
 
    return df

In [17]:
# 6. Entry point 
 
if __name__ == "__main__":
 
    print(f"Device: {DEVICE}")
    print()
 
    # Step 1 — Find best hyperparameters on E047
    best_hp = optuna_search(cell_type=OPTUNA_CELL)
 
    # Step 2 — Train all 56 cell types with best hyperparameters
    results_df = run_all_cell_types(hp=best_hp)
 
    print("\nTop 10 cell types by AUROC:")
    print(results_df.sort_values("test_auroc", ascending=False).head(10).to_string(index=False))

Device: cuda

Running Optuna search on E047 (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]


Optuna complete in 15.2 min
Best val AUROC : 0.8130
Best params    : {'d_model': 128, 'n_heads': 4, 'n_layers': 3, 'ffn_mult': 2, 'dropout': 0.13395652649871614, 'lr': 0.006097025297491433, 'batch_size': 128, 'weight_decay': 8.995191735587168e-06}
Found 56 cell types.
HP: {'d_model': 128, 'n_heads': 4, 'n_layers': 3, 'ffn_mult': 2, 'dropout': 0.13395652649871614, 'lr': 0.006097025297491433, 'batch_size': 128, 'weight_decay': 8.995191735587168e-06}

[ 1/56] E003  AUROC: 0.7742  (48s)
[ 2/56] E004  AUROC: 0.8151  (55s)
[ 3/56] E005  AUROC: 0.8275  (51s)
[ 4/56] E006  AUROC: 0.8228  (51s)
[ 5/56] E007  AUROC: 0.7767  (42s)
[ 6/56] E011  AUROC: 0.7620  (66s)
[ 7/56] E012  AUROC: 0.7919  (46s)
[ 8/56] E013  AUROC: 0.8040  (48s)
[ 9/56] E016  AUROC: 0.7957  (57s)
[10/56] E024  AUROC: 0.7678  (50s)
[11/56] E027  AUROC: 0.8018  (68s)
[12/56] E028  AUROC: 0.8104  (50s)
[13/56] E037  AUROC: 0.8273  (44s)
[14/56] E038  AUROC: 0.8332  (41s)
[15/56] E047  AUROC: 0.8350  (41s)
[16/56] E050  AUROC: 